# Lab 1.2: Few-Shot Prompting for Consistency

**What you'll build:** An extraction system that pulls structured data from varied document formats (invoices, receipts, emails, contracts, handwritten notes) -- and learn why detailed instructions alone produce inconsistent output.

**Estimated time:** 25-30 minutes

| Phase | What happens | Time |
|-------|-------------|------|
| 1 | The Wrong Way -- instructions-only extraction fails on edge cases | 5 min |
| 2 | The Right Way -- few-shot examples with reasoning fix consistency | 8 min |
| 3 | Your Turn -- build your own few-shot examples for new documents | 10 min |
| 4 | Stress Test -- prove your examples generalize to unseen formats | 5 min |

In [ ]:
import anthropic
import json

client = anthropic.Anthropic()  # uses ANTHROPIC_API_KEY env var
MODEL = "claude-sonnet-4-20250514"

## The Setup

You are building a document extraction system for an accounting team. They receive documents in wildly different formats -- formal invoices, crumpled receipts, email threads, legal contracts, and even handwritten notes on scrap paper.

Every document must be extracted into the **same schema**:

```json
{
  "vendor": "str or null",
  "items": [{"name": "str", "price": float}],
  "date": "YYYY-MM-DD or null",
  "total": "float or null",
  "tax": "float or null",
  "payment_method": "str or null",
  "notes": "str or null"
}

```

The hard parts:
- Missing fields must be `null`, not hallucinated
- Informal amounts ("around $278") must be extracted as numbers
- Dates in every format ("03/22/25", "~Mar 10", "January 1, 2025") must become YYYY-MM-DD
- The schema must be exactly the same for every document type

In [ ]:
# The 5 test documents

DOC_INVOICE = """
INVOICE #2024-0892
Acme Corp
Date: March 15, 2025

Bill To: Widget Industries

Items:
  1. Enterprise License (annual)    $12,000.00
  2. Premium Support Package         $3,600.00
  3. Custom Integration Fee          $2,400.00

Subtotal: $18,000.00
Tax (8.25%): $1,485.00
Total Due: $19,485.00

Payment Terms: Net 30
Payment Method: Wire Transfer
"""

DOC_RECEIPT = """
Joe's Hardware
123 Main St

03/22/25  2:34 PM

Hammer          $12.99
Box of Nails     $4.50
Wood Glue        $7.25
---
Subtotal        $24.74
Tax              $2.04
TOTAL           $26.78

VISA ending 4532
"""

DOC_EMAIL = """
From: sarah@vendorcorp.com
To: accounting@mycompany.com
Subject: Re: Payment for consulting work

Hi team,

Attached is the summary for our February engagement:

- Strategy workshop (Feb 3-5): $8,500
- Follow-up report: $2,000
- Travel expenses: $1,200

Total: $11,700. No tax since we're in Oregon.

Please process via ACH to the account on file.

Thanks,
Sarah
"""

DOC_CONTRACT = """
SERVICE AGREEMENT

Between: TechFlow Solutions ("Provider")
And: DataStream Inc. ("Client")

Effective Date: January 1, 2025

Services and Fees:
The Provider shall deliver the following:
(a) Monthly data pipeline maintenance -- $4,500/month
(b) Quarterly performance audit -- $3,000/quarter
(c) Emergency support (as needed) -- $250/hour

First year estimated total: $70,000
(Based on 12 months maintenance + 4 audits + estimated 20 hours emergency)

Payment: Monthly invoices, Net 15
Tax: Subject to applicable state taxes (not included in above figures)
"""

DOC_HANDWRITTEN = """
[Handwritten note, partially legible]

Bought supplies for the Johnson project
~Mar 10

- lumber (about 40 board feet) ... $186
- paint, 3 cans ............... $67
- brushes + misc .............. ~$25

paid cash, no receipt for the misc stuff
total was around $278

need to get reimbursed!!
"""

ALL_DOCS = {
    "invoice": DOC_INVOICE,
    "receipt": DOC_RECEIPT,
    "email": DOC_EMAIL,
    "contract": DOC_CONTRACT,
    "handwritten": DOC_HANDWRITTEN,
}

# Ground truth for scoring
GROUND_TRUTH = {
    "invoice": {
        "vendor": "Acme Corp", "items_count": 3, "date": "2025-03-15",
        "total": 19485.00, "tax": 1485.00, "payment_method": "Wire Transfer",
    },
    "receipt": {
        "vendor": "Joe's Hardware", "items_count": 3, "date": "2025-03-22",
        "total": 26.78, "tax": 2.04, "payment_method": "VISA ending 4532",
    },
    "email": {
        "vendor": "VendorCorp", "items_count": 3, "date": None,
        "total": 11700.00, "tax": 0.0, "payment_method": "ACH",
    },
    "contract": {
        "vendor": "TechFlow Solutions", "items_count": 3, "date": "2025-01-01",
        "total": 70000.00, "tax": None, "payment_method": None,
    },
    "handwritten": {
        "vendor": None, "items_count": 3, "date": "2025-03-10",
        "total": 278.00, "tax": None, "payment_method": "cash",
    },
}

print(f"Loaded {len(ALL_DOCS)} documents: {list(ALL_DOCS.keys())}")
print(f"\nExtraction schema fields: vendor, items, date, total, tax, payment_method, notes")

---

## Phase 1: The Wrong Approach

Most people start with very detailed instructions but no examples. The instructions describe *what* they want, but don't *show* it. Let's see how that goes.

In [ ]:
def build_instructions_only_prompt(document):
    """A detailed prompt with no few-shot examples."""
    return f"""You are a document extraction system. Extract structured data from the 
document below.

Output a JSON object with exactly these fields:
- "vendor": the company or person who issued the document (string or null)
- "items": a list of line items, each with "name" (string) and "price" (float)
- "date": the document date in YYYY-MM-DD format (string or null)
- "total": the total amount (float or null)
- "tax": the tax amount (float or null)
- "payment_method": how payment was made or requested (string or null)
- "notes": any additional relevant information (string or null)

Rules:
- If a field is not present in the document, set it to null
- Do not invent or guess values that aren't in the document
- Convert all dates to YYYY-MM-DD format
- Convert all amounts to plain numbers (no $ signs, no commas)
- For approximate amounts, use the stated number

Document:
---
{document}
---

Return ONLY the JSON object, no other text."""


def extract_document(prompt):
    """Send extraction prompt and parse JSON response."""
    response = client.messages.create(
        model=MODEL,
        max_tokens=1500,
        messages=[{"role": "user", "content": prompt}]
    )
    raw = response.content[0].text.strip()
    if raw.startswith("```"):
        raw = raw.split("\n", 1)[1].rsplit("```", 1)[0].strip()
    try:
        return json.loads(raw)
    except json.JSONDecodeError:
        print(f"Failed to parse JSON. Raw response:\n{raw[:300]}")
        return {}


# Run instructions-only extraction on all 5 documents
print("=== INSTRUCTIONS-ONLY EXTRACTION ===\n")

instructions_results = {}
for doc_type, doc_text in ALL_DOCS.items():
    prompt = build_instructions_only_prompt(doc_text)
    result = extract_document(prompt)
    instructions_results[doc_type] = result
    
    truth = GROUND_TRUTH[doc_type]
    print(f"--- {doc_type.upper()} ---")
    print(f"  Vendor:   {result.get('vendor', 'MISSING')}")
    print(f"  Items:    {len(result.get('items', []))} items")
    print(f"  Date:     {result.get('date', 'MISSING')}")
    print(f"  Total:    {result.get('total', 'MISSING')}")
    print(f"  Tax:      {result.get('tax', 'MISSING')}")
    print(f"  Payment:  {result.get('payment_method', 'MISSING')}")
    print(f"  Notes:    {result.get('notes', 'MISSING')}")
    print()

In [ ]:
# Score the instructions-only approach
def score_extraction(result, truth, doc_type):
    """Score a single extraction against ground truth. Returns dict of checks."""
    checks = {}
    
    # Check vendor
    if truth["vendor"] is None:
        checks["vendor"] = result.get("vendor") is None
    else:
        vendor = result.get("vendor") or ""
        checks["vendor"] = truth["vendor"].lower() in vendor.lower()
    
    # Check items count
    checks["items_count"] = len(result.get("items", [])) == truth["items_count"]
    
    # Check date
    if truth["date"] is None:
        checks["date"] = result.get("date") is None
    else:
        checks["date"] = result.get("date") == truth["date"]
    
    # Check total (within 1% tolerance for float issues)
    if truth["total"] is not None:
        extracted_total = result.get("total")
        if extracted_total is not None:
            checks["total"] = abs(extracted_total - truth["total"]) < truth["total"] * 0.01
        else:
            checks["total"] = False
    
    # Check tax (null handling is critical)
    if truth["tax"] is None:
        checks["tax_null"] = result.get("tax") is None
    elif truth["tax"] == 0.0:
        checks["tax_zero"] = result.get("tax") == 0.0 or result.get("tax") == 0
    else:
        extracted_tax = result.get("tax")
        if extracted_tax is not None:
            checks["tax"] = abs(extracted_tax - truth["tax"]) < 0.01
        else:
            checks["tax"] = False
    
    # Check payment method (null handling)
    if truth["payment_method"] is None:
        checks["payment_null"] = result.get("payment_method") is None
    else:
        pm = result.get("payment_method") or ""
        checks["payment"] = truth["payment_method"].lower() in pm.lower()
    
    return checks


print("=== INSTRUCTIONS-ONLY SCORECARD ===\n")
total_checks = 0
total_passed = 0

for doc_type, result in instructions_results.items():
    truth = GROUND_TRUTH[doc_type]
    checks = score_extraction(result, truth, doc_type)
    passed = sum(1 for v in checks.values() if v)
    total = len(checks)
    total_checks += total
    total_passed += passed
    
    status = "PASS" if passed == total else "FAIL"
    print(f"  {doc_type:<15} {passed}/{total} checks  [{status}]")
    for check_name, check_val in checks.items():
        if not check_val:
            print(f"    FAILED: {check_name}")

instructions_score = total_passed / max(total_checks, 1)
print(f"\n  Overall: {total_passed}/{total_checks} ({instructions_score:.0%})")

### Why did that fail?

- **Null handling is inconsistent.** Without seeing an example of "field not present -> null", the model sometimes returns empty strings, sometimes 0, sometimes omits the field entirely. The contract has "Tax: Subject to applicable state taxes (not included)" -- should that be null or 0? The model guesses differently each run.
- **Approximate values are handled inconsistently.** The handwritten note says "around $278" -- some runs return 278.0, others return null or a string. Without an example showing how to handle approximations, the model improvises.
- **Format varies between runs.** Sometimes the model wraps JSON in markdown code blocks, sometimes it adds commentary. Instructions say "return ONLY JSON" but the model doesn't always comply without seeing the expected format.
- **Edge cases fall through.** The email has "No tax since we're in Oregon" -- is that tax: null or tax: 0? These judgment calls need examples, not rules.

---

## Phase 2: The Right Approach

The fix is **few-shot examples with reasoning**. Instead of describing edge cases in rules, we *show* the model how to handle them.

Key technique: Each example includes **Input -> Output -> Reasoning** so the model learns the *decision logic*, not just the format.

Important: the examples use **different documents** from the test set. We want the model to learn the *pattern*, not memorize answers.

In [ ]:
# Three few-shot examples that cover the critical edge cases

FEW_SHOT_EXAMPLES = [
    {
        "input": """INVOICE INV-2024-445
Bright Solutions LLC
Date: February 20, 2025

To: Greenfield Corp

Web redesign project      $15,000.00
SEO optimization           $3,500.00

Subtotal: $18,500.00
Tax (7%): $1,295.00
Total: $19,795.00

Payment: Check or ACH""",
        "output": {
            "vendor": "Bright Solutions LLC",
            "items": [
                {"name": "Web redesign project", "price": 15000.00},
                {"name": "SEO optimization", "price": 3500.00}
            ],
            "date": "2025-02-20",
            "total": 19795.00,
            "tax": 1295.00,
            "payment_method": "Check or ACH",
            "notes": None
        },
        "reasoning": "Standard invoice. All fields are explicitly stated. Vendor is the issuing company (Bright Solutions LLC), not the recipient. Total includes tax. Payment method is stated directly. No additional notes needed."
    },
    {
        "input": """[Scribbled note on napkin]

Team lunch - friday
~Jan 17

pizzas (3 large) ...... about $45
drinks ................ ~$18
tip ................... $12

paid w/ my personal card
need to expense this""",
        "output": {
            "vendor": None,
            "items": [
                {"name": "pizzas (3 large)", "price": 45.00},
                {"name": "drinks", "price": 18.00},
                {"name": "tip", "price": 12.00}
            ],
            "date": "2025-01-17",
            "total": 75.00,
            "tax": None,
            "payment_method": "personal card",
            "notes": "Approximate amounts. Need to expense."
        },
        "reasoning": "Handwritten/informal note. No vendor name is given (no restaurant named), so vendor is null -- do NOT invent a restaurant name. Approximate amounts ('about $45', '~$18') are converted to their numeric values. Date '~Jan 17' is converted to 2025-01-17 assuming current year. Tax is not mentioned at all, so tax is null -- not 0 (we don't know if tax was included or not). Total is sum of items since no total was explicitly stated. Notes capture the reimbursement request."
    },
    {
        "input": """MAINTENANCE CONTRACT

Provider: CleanPro Services
Client: 500 Oak Street Office Complex
Start Date: March 1, 2025

Monthly cleaning service: $2,800/month
Quarterly deep clean: $1,200/quarter

Annual estimated cost: $38,400
(12 x $2,800 + 4 x $1,200)

Tax: Client responsible for applicable taxes
Payment: Billed monthly""",
        "output": {
            "vendor": "CleanPro Services",
            "items": [
                {"name": "Monthly cleaning service", "price": 2800.00},
                {"name": "Quarterly deep clean", "price": 1200.00}
            ],
            "date": "2025-03-01",
            "total": 38400.00,
            "tax": None,
            "payment_method": None,
            "notes": "Prices are per-period (monthly/quarterly). Annual total is estimated."
        },
        "reasoning": "Contract with recurring fees. Vendor is the provider, not the client. Item prices are per-period rates. Total is the annual estimate given in the document. Tax says 'client responsible' which means tax is not included and not specified -- set to null, NOT 0. Payment says 'billed monthly' which describes billing frequency, not a payment method (no card/ACH/check specified) -- set payment_method to null. Notes clarify that prices are recurring."
    }
]

print(f"Created {len(FEW_SHOT_EXAMPLES)} few-shot examples.")
print(f"\nEdge cases covered:")
print(f"  - Example 1: Standard clean document (baseline)")
print(f"  - Example 2: Informal/handwritten, approximate amounts, null vendor, null tax")
print(f"  - Example 3: Contract with recurring fees, 'tax not included' -> null")

In [ ]:
def build_few_shot_prompt(document, examples):
    """Build extraction prompt with few-shot examples."""
    prompt = """You are a document extraction system. Extract structured data from documents into a consistent JSON format.

Output schema (every extraction must have exactly these fields):
{
  "vendor": "string or null",
  "items": [{"name": "string", "price": float}],
  "date": "YYYY-MM-DD or null",
  "total": "float or null",
  "tax": "float or null",
  "payment_method": "string or null",
  "notes": "string or null"
}

Critical rules:
- If a field is not present or cannot be determined, set it to null
- Do NOT hallucinate values. If the document doesn't name a vendor, vendor is null.
- "Tax not included" or "tax: client responsible" means tax is null (unknown), NOT 0
- "No tax" or "tax exempt" means tax is 0.0 (explicitly zero)
- Approximate amounts ("about $45", "~$18") should be converted to their numeric value
- Billing frequency ("billed monthly") is NOT a payment method. Payment method means HOW payment is made (card, ACH, wire, cash, check).

Here are examples showing how to extract from different document types:

"""
    
    for i, ex in enumerate(examples, 1):
        prompt += f"""=== EXAMPLE {i} ===
INPUT:
{ex['input']}

OUTPUT:
{json.dumps(ex['output'], indent=2)}

REASONING: {ex['reasoning']}

"""
    
    prompt += f"""=== NOW EXTRACT FROM THIS DOCUMENT ===
INPUT:
{document}

OUTPUT:
Return ONLY the JSON object, no other text."""
    
    return prompt


# Run few-shot extraction on all 5 documents
print("=== FEW-SHOT EXTRACTION ===\n")

fewshot_results = {}
for doc_type, doc_text in ALL_DOCS.items():
    prompt = build_few_shot_prompt(doc_text, FEW_SHOT_EXAMPLES)
    result = extract_document(prompt)
    fewshot_results[doc_type] = result
    
    print(f"--- {doc_type.upper()} ---")
    print(f"  Vendor:   {result.get('vendor', 'MISSING')}")
    print(f"  Items:    {len(result.get('items', []))} items")
    print(f"  Date:     {result.get('date', 'MISSING')}")
    print(f"  Total:    {result.get('total', 'MISSING')}")
    print(f"  Tax:      {result.get('tax', 'MISSING')}")
    print(f"  Payment:  {result.get('payment_method', 'MISSING')}")
    print(f"  Notes:    {result.get('notes', 'MISSING')}")
    print()

In [ ]:
# Score and compare
print("=" * 60)
print("COMPARISON: INSTRUCTIONS-ONLY vs FEW-SHOT")
print("=" * 60)

inst_total_checks = 0
inst_total_passed = 0
fs_total_checks = 0
fs_total_passed = 0

print(f"\n{'Document':<15} {'Instructions':>15} {'Few-Shot':>15}")
print(f"{'-'*15} {'-'*15} {'-'*15}")

for doc_type in ALL_DOCS:
    truth = GROUND_TRUTH[doc_type]
    
    inst_checks = score_extraction(instructions_results[doc_type], truth, doc_type)
    inst_p = sum(1 for v in inst_checks.values() if v)
    inst_t = len(inst_checks)
    inst_total_checks += inst_t
    inst_total_passed += inst_p
    
    fs_checks = score_extraction(fewshot_results[doc_type], truth, doc_type)
    fs_p = sum(1 for v in fs_checks.values() if v)
    fs_t = len(fs_checks)
    fs_total_checks += fs_t
    fs_total_passed += fs_p
    
    print(f"{doc_type:<15} {inst_p:>7}/{inst_t:<7} {fs_p:>7}/{fs_t:<7}")

inst_pct = inst_total_passed / max(inst_total_checks, 1)
fs_pct = fs_total_passed / max(fs_total_checks, 1)

print(f"{'-'*15} {'-'*15} {'-'*15}")
print(f"{'TOTAL':<15} {inst_total_passed:>7}/{inst_total_checks:<7} {fs_total_passed:>7}/{fs_total_checks:<7}")
print(f"{'Score':<15} {inst_pct:>14.0%} {fs_pct:>14.0%}")

if fs_pct > inst_pct:
    improvement = fs_pct - inst_pct
    print(f"\nFew-shot examples improved accuracy by {improvement:.0%}.")
    print("The examples taught the model HOW to handle edge cases, not just WHAT fields to extract.")

### Side-by-Side Comparison

| Edge Case | Instructions-Only | Few-Shot |
|-----------|------------------|----------|
| **Handwritten vendor (none)** | Often hallucinated a name | null (correct) |
| **Contract tax ("not included")** | Sometimes 0, sometimes null | null (correct) |
| **Email tax ("no tax, Oregon")** | Sometimes null, sometimes 0 | 0.0 (correct) |
| **Contract payment ("billed monthly")** | Sometimes "monthly" | null (correct -- billing frequency is not a payment method) |
| **Approximate amounts ("~$25")** | Inconsistent handling | 25.00 (correct) |
| **Date formats ("~Mar 10")** | Sometimes failed to parse | 2025-03-10 (correct) |

The key insight: **examples with reasoning teach decision boundaries** that instructions alone cannot.

---

## Phase 3: Your Turn

Here are 3 new documents in formats the examples above didn't cover directly. Build your own set of few-shot examples to extract from these.

**Your goal:** All real fields extracted, missing fields are null (not fabricated), consistent output across all 3 documents.

In [ ]:
# Three new challenge documents

CHALLENGE_DOC_MEDICAL = """
PATIENT BILLING STATEMENT
Valley Health Medical Group

Patient: James Rodriguez
Date of Service: 04/02/2025

Office Visit (Level 3)              $250.00
Blood Panel - Comprehensive           $85.00
Flu Vaccination                       $45.00

Total Charges:                       $380.00
Insurance Adjustment:               -$152.00
Amount Due:                          $228.00

Insurance: Blue Cross PPO
Please remit payment within 30 days
"""

CHALLENGE_DOC_WARRANTY = """
WARRANTY REGISTRATION CARD

Product: PowerMax 3000 Generator
Manufacturer: Titan Power Equipment
Purchase Date: 2025-03-01
Serial #: PM3K-2025-88721

Purchased From: Home Depot, Store #4412
Purchase Price: $1,299.99
Extended Warranty: $149.99

Total Paid: $1,449.98
Tax: included in total

Paid by: Debit Card
"""

CHALLENGE_DOC_SHIPPING = """
SHIPPING MANIFEST / PACKING SLIP

From: Pacific Wholesale Distributors
Order #: PW-2025-1182
Ship Date: Mar 28, 2025

Contents:
  Widget A (500 units) @ $0.45/ea    $225.00
  Widget B (200 units) @ $1.20/ea    $240.00
  Connector Kit (50 units) @ $3.00   $150.00

Merchandise Total: $615.00
Shipping & Handling: $45.00
Order Total: $660.00

Terms: Net 30, PO #AC-7789
Tax: Exempt (reseller certificate on file)
"""

CHALLENGE_DOCS = {
    "medical": CHALLENGE_DOC_MEDICAL,
    "warranty": CHALLENGE_DOC_WARRANTY,
    "shipping": CHALLENGE_DOC_SHIPPING,
}

CHALLENGE_TRUTH = {
    "medical": {
        "vendor": "Valley Health Medical Group",
        "items_count": 3,
        "date": "2025-04-02",
        "total": 228.00,  # Amount Due (after insurance)
        "tax": None,      # No tax mentioned
        "payment_method": None,  # No payment method specified
    },
    "warranty": {
        "vendor": "Home Depot",  # Purchased from, not manufacturer
        "items_count": 2,
        "date": "2025-03-01",
        "total": 1449.98,
        "tax": None,       # "included in total" -- amount unknown
        "payment_method": "Debit Card",
    },
    "shipping": {
        "vendor": "Pacific Wholesale Distributors",
        "items_count": 3,
        "date": "2025-03-28",
        "total": 660.00,
        "tax": 0.0,        # Explicitly tax exempt
        "payment_method": None,  # Net 30 is terms, not a method
    },
}

print("Challenge documents loaded:")
for doc_type in CHALLENGE_DOCS:
    truth = CHALLENGE_TRUTH[doc_type]
    print(f"\n  {doc_type}:")
    print(f"    Key challenges: ", end="")
    if truth["tax"] is None:
        print("tax -> null, ", end="")
    elif truth["tax"] == 0.0:
        print("tax -> 0.0 (exempt), ", end="")
    if truth["payment_method"] is None:
        print("payment -> null, ", end="")
    print()

In [ ]:
# =============================================================
# TODO: Build your own few-shot examples
# =============================================================
#
# Requirements:
# - 3-4 examples (do NOT use the challenge documents as examples)
# - At least one example with null fields (missing data)
# - At least one example with an edge case like:
#     - "tax included in total" (tax is null, not 0)
#     - "tax exempt" (tax is 0.0)
#     - Insurance adjustments
#     - Payment terms vs payment method distinction
# - Each example needs: input, output, reasoning
#
# Think about:
# - What edge cases in the challenge docs are NOT covered
#   by the Phase 2 examples?
# - How do you teach the model the difference between
#   "tax included" (null) vs "tax exempt" (0.0)?
# - Is the vendor the manufacturer or the seller?

YOUR_EXAMPLES = [
    # TODO: Example 1
    {
        "input": """YOUR EXAMPLE DOCUMENT HERE""",
        "output": {
            "vendor": None,
            "items": [],
            "date": None,
            "total": None,
            "tax": None,
            "payment_method": None,
            "notes": None
        },
        "reasoning": "YOUR REASONING HERE"
    },
    # TODO: Example 2
    {
        "input": """YOUR EXAMPLE DOCUMENT HERE""",
        "output": {
            "vendor": None,
            "items": [],
            "date": None,
            "total": None,
            "tax": None,
            "payment_method": None,
            "notes": None
        },
        "reasoning": "YOUR REASONING HERE"
    },
    # TODO: Example 3
    {
        "input": """YOUR EXAMPLE DOCUMENT HERE""",
        "output": {
            "vendor": None,
            "items": [],
            "date": None,
            "total": None,
            "tax": None,
            "payment_method": None,
            "notes": None
        },
        "reasoning": "YOUR REASONING HERE"
    },
]

# Run your examples against the challenge documents
print("=== YOUR FEW-SHOT EXTRACTION ===\n")

your_results = {}
for doc_type, doc_text in CHALLENGE_DOCS.items():
    prompt = build_few_shot_prompt(doc_text, YOUR_EXAMPLES)
    result = extract_document(prompt)
    your_results[doc_type] = result
    
    print(f"--- {doc_type.upper()} ---")
    print(f"  Vendor:   {result.get('vendor', 'MISSING')}")
    print(f"  Items:    {len(result.get('items', []))} items")
    print(f"  Date:     {result.get('date', 'MISSING')}")
    print(f"  Total:    {result.get('total', 'MISSING')}")
    print(f"  Tax:      {result.get('tax', 'MISSING')}")
    print(f"  Payment:  {result.get('payment_method', 'MISSING')}")
    print(f"  Notes:    {result.get('notes', 'MISSING')}")
    print()

In [ ]:
# =============================================================
# Checker: validates your solution
# =============================================================

print("=== YOUR SCORECARD ===\n")

your_total_checks = 0
your_total_passed = 0
all_passed = True

for doc_type, result in your_results.items():
    truth = CHALLENGE_TRUTH[doc_type]
    checks = score_extraction(result, truth, doc_type)
    passed = sum(1 for v in checks.values() if v)
    total = len(checks)
    your_total_checks += total
    your_total_passed += passed
    
    status = "PASS" if passed == total else "NEEDS WORK"
    if passed < total:
        all_passed = False
    print(f"  {doc_type:<15} {passed}/{total} checks  [{status}]")
    for check_name, check_val in checks.items():
        if not check_val:
            print(f"    FAILED: {check_name}")
            # Provide hints
            if "null" in check_name:
                print(f"      Hint: Add an example showing this field as null")
            if "tax" in check_name:
                print(f"      Hint: 'tax included' -> null, 'tax exempt' -> 0.0")
            if "payment" in check_name:
                print(f"      Hint: Payment terms (Net 30) != payment method (card, ACH)")

your_pct = your_total_passed / max(your_total_checks, 1)
print(f"\n  Overall: {your_total_passed}/{your_total_checks} ({your_pct:.0%})")

if all_passed:
    print("\n  PASSED -- Consistent extraction across all challenge documents!")
    print("  Your few-shot examples successfully taught the edge cases.")
else:
    print("\n  Revise your examples. Each failed check tells you what edge case")
    print("  your examples don't cover yet.")

### Success Criteria

The checker above tests:

1. **All real fields extracted** -- vendor, items, date, total must match ground truth
2. **Missing fields are null, not fabricated** -- medical has no payment method, shipping has no payment method, warranty tax is null ("included" is not a number)
3. **Tax edge cases handled correctly** -- "included in total" -> null, "exempt" -> 0.0, not mentioned -> null
4. **Payment method vs payment terms** -- "Net 30" is a term, not a method; "Debit Card" is a method

If you're failing null checks, add an example where a similar field is explicitly null with reasoning explaining *why*.

If you're failing tax checks, make sure one example shows "tax included" -> null and another shows "tax exempt" -> 0.0.

---

## Phase 4: Stress Test

Good few-shot examples should generalize to documents the examples never covered. Let's test with unusual formats and verify consistency across multiple runs.

In [ ]:
# Run your examples 5x against the medical document (hardest edge case)
print("Running your prompt 5 times against the medical billing statement...\n")

medical_runs = []
for run in range(5):
    prompt = build_few_shot_prompt(CHALLENGE_DOC_MEDICAL, YOUR_EXAMPLES)
    result = extract_document(prompt)
    checks = score_extraction(result, CHALLENGE_TRUTH["medical"], "medical")
    passed = sum(1 for v in checks.values() if v)
    total = len(checks)
    medical_runs.append({
        "passed": passed, "total": total,
        "tax": result.get("tax"), "payment": result.get("payment_method"),
        "total_amount": result.get("total")
    })
    print(f"  Run {run+1}: {passed}/{total} checks | tax={result.get('tax')} | payment={result.get('payment_method')} | total={result.get('total')}")

# Check consistency
tax_values = [r["tax"] for r in medical_runs]
payment_values = [r["payment"] for r in medical_runs]
total_values = [r["total_amount"] for r in medical_runs]

print(f"\n=== CONSISTENCY CHECK ===")
print(f"Tax values across 5 runs:     {tax_values}")
print(f"Payment values across 5 runs: {payment_values}")
print(f"Total values across 5 runs:   {total_values}")

tax_consistent = len(set(str(v) for v in tax_values)) == 1
payment_consistent = len(set(str(v) for v in payment_values)) == 1
total_consistent = len(set(str(v) for v in total_values)) == 1

if tax_consistent and payment_consistent and total_consistent:
    print("\nPerfect consistency -- your examples produce the same results every run.")
else:
    if not tax_consistent:
        print("\nTax is inconsistent. Add a clearer example for 'no tax mentioned' -> null.")
    if not payment_consistent:
        print("\nPayment is inconsistent. Clarify when payment_method should be null.")
    if not total_consistent:
        print("\nTotal varies. Add an example showing how to handle insurance adjustments.")

In [ ]:
# Edge case: a document format that NO example covered
# Good few-shot examples generalize; bad ones don't

EDGE_CASE_DOCS = {
    "text_message": {
        "doc": """
[Text message screenshot]

Mike: hey can you grab lunch supplies for the offsite?
Me: sure, got everything at costco
Me: sandwiches platter $89, chips and dip $34, drinks $52
Me: they didnt have the cookies so skipped that
Me: total was $175 (no tax on food here)
Me: paid with company amex
Mike: perfect thx
""",
        "expected_vendor": "Costco",
        "expected_items": 3,
        "expected_tax": 0.0,  # "no tax" = explicitly 0
        "expected_payment": True,  # Should capture "company Amex"
    },
    "foreign_currency": {
        "doc": """
RECHNUNG (Invoice)
Schmidt & Weber GmbH
Datum: 15. Marz 2025

Beratungsleistungen (Consulting)    EUR 4.500,00
Reisekosten (Travel)                EUR 890,00

Netto: EUR 5.390,00
MwSt 19%: EUR 1.024,10
Gesamt: EUR 6.414,10

Bankuberweisung an: IBAN DE89 3704 0044 0532 0130 00
""",
        "expected_vendor": "Schmidt",
        "expected_items": 2,
        "expected_tax": True,  # Should capture MwSt
        "expected_payment": True,  # Bank transfer
    },
}

print("=== EDGE CASE: Unseen document formats ===\n")

for case_name, case_data in EDGE_CASE_DOCS.items():
    prompt = build_few_shot_prompt(case_data["doc"], YOUR_EXAMPLES)
    result = extract_document(prompt)
    
    print(f"--- {case_name.upper()} ---")
    print(f"  Vendor:   {result.get('vendor', 'MISSING')}")
    print(f"  Items:    {len(result.get('items', []))} items (expected {case_data['expected_items']})")
    print(f"  Tax:      {result.get('tax', 'MISSING')}")
    print(f"  Payment:  {result.get('payment_method', 'MISSING')}")
    
    # Quick checks
    issues = []
    if case_data["expected_vendor"]:
        vendor = result.get("vendor") or ""
        if case_data["expected_vendor"].lower() not in vendor.lower():
            issues.append(f"vendor mismatch (expected '{case_data['expected_vendor']}')")
    if len(result.get("items", [])) != case_data["expected_items"]:
        issues.append(f"items count mismatch")
    if case_data["expected_tax"] == 0.0 and result.get("tax") != 0.0 and result.get("tax") != 0:
        issues.append(f"tax should be 0.0")
    if case_data.get("expected_payment") and result.get("payment_method") is None:
        issues.append("payment_method should not be null")
    
    if issues:
        print(f"  Issues: {issues}")
    else:
        print(f"  Looks good -- your examples generalized to this unseen format.")
    print()

print("If your examples generalized well, they taught the MODEL the decision logic,")
print("not just the format. That's the goal of good few-shot design.")

### Key Takeaways

1. **Instructions describe what you want. Examples show how to get it.** For edge cases (null handling, approximate values, format ambiguity), examples with reasoning are far more effective than more rules.
2. **Include reasoning in your examples.** The model learns the *decision logic* ("tax is null because 'included in total' means the amount is unknown"), not just the surface pattern.
3. **Cover edge cases explicitly.** One example showing a null field teaches the model more about null handling than ten lines of instructions about it.
4. **Good examples generalize.** If your examples teach *principles* (how to decide when something is null vs 0 vs missing), the model can handle document types it's never seen before.